# 29 — Gateway Prediction Percentages (BPIC17)

Given a **gateway activity**, collect every test prefix that ends at that activity and ask the model what it predicts next. Report:

- **Argmax distribution** — for how many prefixes each activity was the top-1 prediction (% of prefixes).
- **Mean softmax** — the average probability the model assigned to each activity across those prefixes.
- **Mean top-1 confidence** — how sure the model was on average.

The decoder softmax is the point-estimate next-step distribution (dropout disabled). Use notebook 22 for a decision-tree surrogate view.

## 1. Setup

In [ ]:
import sys
from pathlib import Path

_current = Path().resolve()
while _current != _current.parent:
    if (_current / 'src').is_dir():
        break
    _current = _current.parent

if str(_current) not in sys.path:
    sys.path.insert(0, str(_current))
src_path = str(_current / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.model.dropout_uncertainty_enc_dec_LSTM.dropout_uncertainty_model import DropoutUncertaintyEncoderDecoderLSTM
from src.interpretability.utils.tensor_decoder import TensorDecoder
from src.interpretability.perturbation_methods.revised_plus.revised_plus import RevisedPlusModelPredictor

# --- Dataset ---
data_path = _current / 'encoded_data' / 'BPIC_2017_all_5_test.pkl'
dataset = torch.load(data_path, weights_only=False)

sample = dataset[0]
n_cat = len(sample[0])
n_num = len(sample[1])
seq_len = sample[0][0].shape[0]
print(f'Dataset: {len(dataset)} sequences, {n_cat} cat, {n_num} num, seq_len={seq_len}')

# --- Model (same as notebook 22) ---
from src.interpretability.config.bpic17_config import CONFIG
CONFIG.use_improved = False  # set True for the improved variant
model = DropoutUncertaintyEncoderDecoderLSTM.load(str(CONFIG.get_model_path()), dropout=0.0)
model.eval()

decoder = TensorDecoder(dataset)

ACTIVITY_FEATURE = 'concept:name'
activity_idx_to_label = decoder.idx_to_label[ACTIVITY_FEATURE]
max_idx = max(activity_idx_to_label.keys())
activity_names = [activity_idx_to_label.get(i, f'<unk_{i}>') for i in range(max_idx + 1)]
eos_idx = next(i for i, name in enumerate(activity_names) if name == 'EOS')

predictor = RevisedPlusModelPredictor(model, suffix_step=0, activity_feature=ACTIVITY_FEATURE)

ACT_IDX = decoder.cat_features.index(ACTIVITY_FEATURE)
print(f'Activity vocabulary ({len(activity_names)}), EOS={eos_idx}, ACT_IDX={ACT_IDX}')

## 2. Choose the gateway

Set `GATEWAY_ACTIVITY` to the activity whose outgoing decision you want to inspect. BPIC17 has many candidates — try e.g. `W_Validate application`, `A_Create Application`, `O_Create Offer`, `W_Call after offers`, `A_Submitted`, `O_Sent (mail and online)`, etc. The first cell below prints the most common activities so you can pick one.

In [ ]:
# Quick look at which activities have many outgoing decisions in the test data.
cases_tmp = {}
for i in range(len(dataset)):
    cat_t, _, case_id = dataset[i]
    tl = int((cat_t[0] != 0).sum().item())
    if case_id not in cases_tmp or tl > cases_tmp[case_id][1]:
        cases_tmp[case_id] = (i, tl)

from collections import Counter
succ = Counter()
for cid, (ix, tl) in cases_tmp.items():
    cat_full, _, _ = dataset[ix]
    s = seq_len - tl
    acts = cat_full[ACT_IDX][s:s + tl]
    useful = tl
    for j in range(tl - 1, -1, -1):
        if acts[j].item() == eos_idx:
            useful = j
        else:
            break
    for k in range(useful - 1):
        succ[activity_names[acts[k].item()]] += 1

print('Top activities by outgoing transitions in the test set:')
for a, c in succ.most_common(20):
    print(f'  {a:<40s}  {c}')

In [ ]:
GATEWAY_ACTIVITY = 'W_Validate application'  # <-- change this
TOP_K = 15                                    # how many entries to show in the ranking

## 3. Collect test prefixes ending at this activity

In [ ]:
cases = {}
for i in range(len(dataset)):
    cat_t, num_t, case_id = dataset[i]
    trace_len = int((cat_t[0] != 0).sum().item())
    if case_id not in cases or trace_len > cases[case_id][1]:
        cases[case_id] = (i, trace_len)

cat_list = []
num_list = []
actual_next = []

for case_id, (ds_idx, trace_len) in cases.items():
    cat_full, num_full, _ = dataset[ds_idx]
    src_start = seq_len - trace_len
    acts = cat_full[ACT_IDX][src_start:src_start + trace_len]

    useful_len = trace_len
    for j in range(trace_len - 1, -1, -1):
        if acts[j].item() == eos_idx:
            useful_len = j
        else:
            break

    for k in range(useful_len - 1):
        if activity_names[acts[k].item()] != GATEWAY_ACTIVITY:
            continue
        prefix_len = k + 1
        pad_len = seq_len - prefix_len

        cat_prefix = []
        for c in cat_full:
            t = torch.zeros_like(c)
            t[pad_len:] = c[src_start:src_start + prefix_len]
            cat_prefix.append(t)
        num_prefix = []
        for n in num_full:
            t = torch.zeros_like(n)
            t[pad_len:] = n[src_start:src_start + prefix_len]
            num_prefix.append(t)

        cat_list.append(cat_prefix)
        num_list.append(num_prefix)
        actual_next.append(activity_names[acts[k + 1].item()])

N = len(cat_list)
print(f"Gateway '{GATEWAY_ACTIVITY}': {N} test prefixes found.")
assert N > 0, f'No test prefixes end at activity {GATEWAY_ACTIVITY!r}'

## 4. Run the model (batched)

In [ ]:
cat_batch = [torch.stack([cat_list[i][ci] for i in range(N)]) for ci in range(n_cat)]
num_stacked = torch.stack([torch.stack(num_list[i], dim=-1) for i in range(N)])

preds, probs = predictor.predict_batch(cat_batch, num_stacked, batch_size=64)
print('preds shape:', preds.shape, '| probs shape:', probs.shape)

## 5. Percentages

- **Argmax %** — fraction of the N prefixes where this activity was top-1.
- **Mean softmax %** — mean probability the model assigned to this activity.
- **Actual %** — what actually happened next in the log.

In [ ]:
n_classes = probs.shape[1]
class_names = [activity_names[i] if i < len(activity_names) else f'<cls_{i}>' for i in range(n_classes)]

argmax_counts = np.bincount(preds, minlength=n_classes)
argmax_pct = argmax_counts / N * 100
mean_probs_pct = probs.mean(axis=0) * 100

actual_counts = {a: 0 for a in class_names}
for a in actual_next:
    actual_counts[a] = actual_counts.get(a, 0) + 1
actual_pct = np.array([actual_counts.get(n, 0) for n in class_names]) / N * 100

mean_top1 = probs.max(axis=1).mean() * 100

df = pd.DataFrame({
    'Activity': class_names,
    'Argmax %': argmax_pct,
    'Mean softmax %': mean_probs_pct,
    'Actual %': actual_pct,
})
df = df[df[['Argmax %', 'Mean softmax %', 'Actual %']].sum(axis=1) > 0]
df = df.sort_values('Mean softmax %', ascending=False).reset_index(drop=True)

print(f"Gateway: {GATEWAY_ACTIVITY}")
print(f"N prefixes: {N}")
print(f"Mean top-1 confidence: {mean_top1:.1f}%\n")

with pd.option_context('display.float_format', '{:.2f}'.format):
    display(df.head(TOP_K))

In [ ]:
top = df.head(TOP_K).iloc[::-1]
y = np.arange(len(top))
h = 0.28

fig, ax = plt.subplots(figsize=(11, max(3.5, len(top) * 0.45)))
ax.barh(y + h, top['Argmax %'], h, label='Argmax %', color='steelblue')
ax.barh(y,     top['Mean softmax %'], h, label='Mean softmax %', color='seagreen')
ax.barh(y - h, top['Actual %'], h, label='Actual %', color='darkorange')
ax.set_yticks(y)
ax.set_yticklabels(top['Activity'])
ax.set_xlabel('%')
ax.set_title(f"Predictions after '{GATEWAY_ACTIVITY}'  —  N={N} test prefixes")
ax.legend()
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Plain text list

In [ ]:
print(f"Argmax predictions after '{GATEWAY_ACTIVITY}' (N={N}):")
for _, r in df.iterrows():
    if r['Argmax %'] > 0:
        print(f"  {r['Activity']:<40s}  {r['Argmax %']:6.2f}%")

print(f"\nMean softmax probability after '{GATEWAY_ACTIVITY}' (N={N}):")
for _, r in df.iterrows():
    if r['Mean softmax %'] >= 0.1:
        print(f"  {r['Activity']:<40s}  {r['Mean softmax %']:6.2f}%")